# CCS+Q for NaCl
In this tutorial we fit a CCS potential alongside a point charge description with effective atomic charges unique to each element. The reference data in the example is generated from an anyltical two-body potential. The functional form is given in the dictionary `G2B_params`. The analytical form of the potential is given by `"V_func"` where more or less any reasonalbe function expressed in terms of `r_ij` can be used. 

In [1]:
from ase.io import read,write
from ase.build import bulk
import numpy as np
import ase.db as db
from ase.visualize import view
from ase.calculators.emt import EMT 
from ccs_fit.ase_calculator.ccs_ase_G2B import G2B
import matplotlib.pyplot as plt
import json

NaCl=bulk('NaCl','rocksalt',cubic=True,a=5.6)
NaCl=NaCl*[2,2,2]

G2B_params={
        "Charges": {
                "Na": 1.0,
                "Cl": -1.0
        },
        "One_body": {
                "Na": 0.0,
                "Cl": 0.0
        },
        "Two_body": {
                "Na-Na": {
                        "r_min": 0.0,
                        "r_cut": 8.0,
                        "V_func": " 0.2636971851926334*exp( (2.34- r_ij) /0.317)      -1.0485876722813794/(r_ij**6)-0.4993377877804923/(r_ij**8)"
                },
                "Na-Cl": {
                        "r_min": 0.0,
                        "r_cut": 8.0,
                        "V_func": " 0.21096642097716847*exp( (2.755- r_ij) /0.317)      -6.990512208350348/(r_ij**6)-8.675685093364478/(r_ij**8)"
                },
                "Cl-Cl": {
                        "r_min": 0.0,
                        "r_cut": 8.0,
                        "V_func": " 0.15823565676170354*exp( (3.17- r_ij) /0.317)      -72.40150747359878/(r_ij**6)-145.427198022952/(r_ij**8)"
                }
        }
}
calc = G2B(G2B_params)
NaCl.calc = calc


### Generate training data
Curvature Constrained Splines can be fitted to a reference data-set with energies (and optionally forces) of pre-calculated structures. 
> **Note:** Stresses cannot be computed in this example since we use the `pymatgen` library to compute the energy and forces arising from the point charges. However, `pymatgen` does not provide stress. 

We first define the function to generate the training-data.

In [2]:
def do_training():
    orig_cell = NaCl.get_cell()
    orig_struc = NaCl.copy()
    Emax=0.0 # Maximum energy to include in the database
    displacement_magnitude=0.02
    disp_steps=8
    scale_steps=10

    counter=1
    total_iterations=scale_steps*disp_steps
    with tqdm(total=total_iterations) as pbar:
        for scale in np.linspace(0.95, 1.15, scale_steps):
            new_cell = orig_cell*scale
            new_struc = orig_struc.copy()
            new_struc.set_cell(new_cell)
            new_struc.calc = calc
            nrg = new_struc.get_potential_energy()
            for i in range(disp_steps):
                rattle_struc = new_struc.copy()
                rattle_struc.rattle(displacement_magnitude*i, seed=counter)
                rattle_struc.calc = calc
                nrg = rattle_struc.get_potential_energy()
                if nrg < Emax: # We exclude structures that are unreasonably high in energy
                    xyz_file=f"CALCULATED_DATA/S{counter}.xyz"
                    write(xyz_file,rattle_struc)
                    counter += 1
                pbar.update(1)

       

Next we generate the actual data for our training set. Since he training takes considerable time, you can skip this step and use the pre-computed data instead. 

In [ ]:
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm import tqdm


# Cleanup function triggered by button click
def train(b):
    with out:
        clear_output(wait=True)
        current_directory = os.getcwd()
        for file_path in glob.glob(os.path.join(current_directory, 'CALCULATED_DATA/*')):
            os.remove(file_path)
        do_training()
        print("Training-set completed.")

# Function to cancel cleanup
def cancel_train(b):
    with out:
        clear_output(wait=True)
        print("Skipping generating training-set.")

# Create Yes/No buttons
button_yes = widgets.Button(description="Yes", button_style='danger')
button_no = widgets.Button(description="No", button_style='success')

button_yes.on_click(train)
button_no.on_click(cancel_train)

# Display prompt
print("Would you like to re-create the traning-set? (Note that this timeconsuming and will overwrite the existing training-set)")

display(button_yes, button_no)

# Output area for messages
out = widgets.Output()
display(out)

### Building a reference database

After generating the data, we collect it in an ASE database file. The ``file_list`` is a file containing a list of files to be collected into the data base.

Example of a ``file_list`` file:

    CALCULATED_DATA/S1.xyz
    CALCULATED_DATA/S2.xyz
    CALCULATED_DATA/S3.xyz
    CALCULATED_DATA/S4.xyz
    
Any format supported by ASE can be read in.

In [4]:
# Write the list of files to a file
import glob

f = open("file_list", "w")
current_directory = os.getcwd()

for file_path in glob.glob(os.path.join(current_directory, 'CALCULATED_DATA/*')):
    print(file_path,file=f)
f.close()

In [ ]:
from ccs_fit.scripts.ccs_build_db import ccs_build_db

ccs_build_db(mode="CCS",DFT_DB="NaCl.db",file_list="file_list",overwrite=True)

### Fit training data to Curvature Constrained Splines + Point charges
Finally, the splines are fitted to the target defined in the `NaCl.db`  file. 

> **Note:** Stresses cannot be fitted in this example since we use the `pymatgen` library to compute the energy and forces arising from the point charges. However, `pymatgen` does not provide stress. In order to fit using stress we have to set `"EwaldRoutine" : "lammps"` in the input to `ccs_fit`. This requires you to have `LAMMPS` installed and properly connected to `ASE` since `ccs_fit` make use of the ase lammps calculator.



In [3]:
### Generate input.json file
import json

input={
        "General": {
                "Interface": "CCS+Q",
                "FitForces": "True",
                "FitStresses": "False"
        },
        "TrainSet": "NaCl.db",
        "Twobody": {
                "Na-Na": {
                        "Rcut": 8.0,
                        "Resolution": 0.05,
                        "SwType": "sw",
                        "SearchMode": "Sparse",
                        "SearchResolution": 0.5
                },
                "Na-Cl": {
                        "Rcut": 8.0,
                        "Resolution": 0.05,
                        "SwType": "sw",
                        "SearchMode": "Sparse",
                        "SearchResolution": 0.5
                },
                "Cl-Cl": {
                        "Rcut": 8.0,
                        "Resolution": 0.05,
                        "SwType": "sw",
                        "SearchMode": "Sparse",
                        "SearchResolution": 0.5
                }

        },
        "Onebody": [
                "Na",
                "Cl"
        ],
        "Charges": {
                "Na": 1.0,
                "Cl":-1.0
        },
        "EwaldRoutine" : "pymatgen",
}
#SAVE TO FILE
with open('CCS_input.json', 'w') as f:
    json.dump(input, f, indent=8)

The `"SearchMode" : "sparse"` option allow us to limit the search for inflection points in the curvature. This save alot of time but is less accurate. 

In [ ]:
from ccs_fit import ccs_fit

ccs_fit("CCS_input.json")

### Validate your potential
Make sure your potential (at least) reproduce the data points in your training-set. Performing further tests on strucutres not included in the training set is recomended but not included in the tutorial.

In [ ]:
from ccs_fit.scripts.ccs_validate import ccs_validate
ccs_validate(mode="CCS",CCS_params="CCS_params.json",DFT_DB="NaCl.db")

In [ ]:
from ccs_fit.ase_calculator.ccs_ase_calculator import spline_table
from ccs_fit.ase_calculator.ccs_ase_G2B import G2B_pair

with open("CCS_params.json", "r") as f:
    CCS_params = json.load(f)

for pair in CCS_params["Two_body"]:
    #plt.ylim(-0.25,2.0)
    r=np.arange(CCS_params["Two_body"][pair]["r_min"], CCS_params["Two_body"][pair]["r_cut"], 0.01)
    elem1, elem2 = pair.split("-")
    tb = spline_table(elem1, elem2, CCS_params)
    gtb = G2B_pair(elem1,elem2,G2B_params)
    y1 = [tb.eval_energy(rs) for rs in r]
    y2 = [gtb.eval_energy(rs) for rs in r]
    plt.plot(r,y2,color='black',label=f"{pair} Reference")
    plt.plot(r,y1,'--',color='red',label=f"{pair} CCS")
    plt.xlabel('Distance (Å)')
    plt.ylabel('Energy (eV)')
    plt.legend()
    plt.show()

err=np.loadtxt("CCS_validate.dat")
err[:,0]=err[:,0]/err[:,3]
err[:,1]=err[:,1]/err[:,3]
plt.xlabel('Reference energy (eV/atom)')
plt.ylabel('Validation energy (eV/atom)')
plt.plot( [min(err[:,0]),max(err[:,0])],[min(err[:,0]),max(err[:,0])],'--',color='black'  )
plt.scatter(err[:,0],err[:,1],facecolors='none', edgecolors='red')
plt.show()
plt.xlabel('Reference energy (eV/atom)')
plt.ylabel('Error in fit (eV/atom)')
plt.scatter(err[:,0],err[:,2],facecolors='none', edgecolors='red')
plt.show()

try:
    err_F=np.loadtxt("CCS_error_forces.out")
    plt.xlabel('Reference force (eV/Å)')
    plt.ylabel('Fitted force (eV/Å)')
    plt.plot( [min(err_F[:,0]),max(err_F[:,0])],[min(err_F[:,0]),max(err_F[:,0])],'--',color='black')
    plt.scatter(err_F[:,0],err_F[:,1],facecolors='none', edgecolors='red',alpha=0.1 )
    plt.show()
except:
    pass


### We can also inspect the values of the fitted charges
In this case we know the reference value allowing us to compare.

In [ ]:
for item in CCS_params["Charges"]:
    print(f"Fitted charge of {item} {CCS_params['Charges'][item]} vs target {G2B_params['Charges'][item]}")


# Export FF
After checking the quality of our fitted potential we can export it for use with e.g. lammps.

You can for example export the potential to the *uf3*-format using the following commands. A specification to be used in the lammps input will be printed to screen and the parameters will be saved to a file called `CCS.uf3`.

In [ ]:
from ccs_fit.scripts.ccs_export_FF import ccs_export_FF
ccs_export_FF("CCS_params.json",form="lammps_uf3")


You can for also export the potential to the *table*-format using the following commands. A specification to be used in the lammps input will be printed to screen and the parameters will be saved to a file called `CCS.table`.

In [ ]:
ccs_export_FF("CCS_params.json")

# Cleaning up

In [ ]:
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define the directories
current_directory = os.getcwd()

# Function to remove files except for the specified file
def remove_files_except(directory, exception_file):
    for file_path in glob.glob(os.path.join(directory, '*')):
        if os.path.isfile(file_path) and not file_path.endswith(exception_file):
            os.remove(file_path)

# Cleanup function triggered by button click
def cleanup(b):
    with out:
        clear_output(wait=True)
        remove_files_except(current_directory, 'ipynb')
        print("Cleanup completed.")

# Function to cancel cleanup
def cancel_cleanup(b):
    with out:
        clear_output(wait=True)
        print("No changes made.")

# Create Yes/No buttons
button_yes = widgets.Button(description="Yes", button_style='danger')
button_no = widgets.Button(description="No", button_style='success')

button_yes.on_click(cleanup)
button_no.on_click(cancel_cleanup)

# Display prompt
print("Would you like to clean up?")
display(button_yes, button_no)

# Output area for messages
out = widgets.Output()
display(out)
